In [1]:
# ============================================================
# BVMT News Scraper — ilboursa.com  FULL HISTORICAL VERSION
# Scrapes ALL pages for every stock from 2016 to today.
#
# HOW PAGINATION WORKS ON ILBOURSA:
#   Page 1:  https://www.ilboursa.com/marches/news_valeur?s=AB
#   Page 2:  https://www.ilboursa.com/marches/news_valeur?p=2&s=AB
#   Page N:  https://www.ilboursa.com/marches/news_valeur?p=N&s=AB
#
# STOP CONDITION:
#   The ">" (next) arrow disappears on the last page.
#   We detect this by checking if a link with text ">" exists.
#   When it is gone → we have scraped all pages → stop.
#
# Copy each section into a separate Jupyter cell.
# ============================================================
 

In [7]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 1 — Imports                                        ║
# ╚══════════════════════════════════════════════════════════╝
 
import requests
from bs4 import BeautifulSoup
import psycopg2
import psycopg2.extras
import pandas as pd
from dotenv import load_dotenv
from datetime import datetime, date
from urllib.parse import quote, urljoin
import time
import os
import re
 
load_dotenv()
 
def get_conn():
    return psycopg2.connect(
        host=os.getenv('DB_HOST'),
        port=int(os.getenv('DB_PORT', 5432)),
        dbname=os.getenv('DB_NAME'),
        user=os.getenv('DB_USER'),
        password=os.getenv('DB_PASSWORD')
    )
 
print("Imports OK")
 

Imports OK


In [2]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 2 — Ticker mapping (68 stocks, all verified)       ║
# ╚══════════════════════════════════════════════════════════╝
 
ILBOURSA_TICKERS = {
    # ── Banking ──────────────────────────────────────────────
    'AMEN BANK':         'AB',
    'BIAT':              'BIAT',
    'ATTIJARI BANK':     'TJARI',
    'BH':                'BH',
    'BNA':               'BNA',
    'BT':                'BT',
    'STB':               'STB',
    'UIB':               'UIB',
    'ATB':               'ATB',
    'UBCI':              'UBCI',
    'WIFACK INT BANK':   'WIFAK',
    # ── Insurance ────────────────────────────────────────────
    'STAR':              'STAR',
    'ASTREE':            'AST',
    'BH ASSURANCE':      'BHASS',
    'TUNIS RE':          'TRE',
    # ── Leasing / Finance ────────────────────────────────────
    'TUNISIE LEASING F': 'TLS',
    'ATTIJARI LEASING':  'TJL',
    'HANNIBAL LEASE':    'HL',
    'BEST LEASE':        'BL',
    'MODERN LEASING':    'BHL',
    'TUNISIE VALEURS':   'TVAL',
    'SPDIT - SICAF':     'SPDIT',
    'PLAC. TSIE-SICAF':  'PLTU',
    'TUNINVEST-SICAR':   'TINV',
    'BTE (ADP)':         'BTE',
    'SIMPAR':            'SIMPA',
    # ── Industry / Food / Beverage ───────────────────────────
    'SFBT':              'SFBT',
    'DELICE HOLDING':    'DH',
    'POULINA GP HOLDING':'PGH',
    'ALKIMIA':           'ALKIM',
    'ADWYA':             'ADWYA',
    'SITS':              'SITS',
    'SOTIPAPIER':        'STPAP',
    'SOMOCER':           'SOMOC',
    'SOTUMAG':           'MGR',
    'SOTUVER':           'SOTUV',
    'SOTRAPIL':          'STPIL',
    'SIPHAT':            'SIPHA',
    'UNIMED':            'UMED',
    'MAGASIN GENERAL':   'MAG',
    'MONOPRIX':          'MNP',
    # ── Technology / Telecom ─────────────────────────────────
    'ONE TECH HOLDING':  'OTH',
    'SOTETEL':           'SOTET',
    'CELLCOM':           'CELL',
    'TELNET HOLDING':    'TLNET',
    'GIF-FILTER':        'GIF',
    # ── Auto / Transport ─────────────────────────────────────
    'ENNAKL AUTOMOBILES':'NAKL',
    'CITY CARS':         'CC',
    'EURO-CYCLES':       'ECYCL',
    'TUNISAIR':          'TAIR',
    'ARTES':             'ARTES',
    # ── Construction / Materials ─────────────────────────────
    'CIMENTS DE BIZERTE':'SCB',
    'ELBENE INDUSTRIE':  'ELBEN',
    'SAH':               'SAH',
    'SIAME':             'SIAME',
    'ELECTROSTAR':       'LSTR',
    'ASSAD':             'ASSAD',
    # ── Other ────────────────────────────────────────────────
    'ICF':               'ICF',
    'CIL':               'CIL',
    'ATL':               'ATL',
    'TPR':               'TPR',
    'SOPAT':             'SOPAT',
    'MPBS':              'MPBS',
    'UADH':              'UADH',
    'STEQ':              'STEQ',
    'AIR LIQUDE TSIE':   'AL',
    'ATELIER MEUBLE INT':'SAM',
    'ESSOUKNA':          'SOKNA',
}
 
print(f"Ticker mapping: {len(ILBOURSA_TICKERS)} stocks")
 

Ticker mapping: 68 stocks


In [3]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 3 — Session with browser headers                   ║
# ╚══════════════════════════════════════════════════════════╝
 
SESSION = requests.Session()
SESSION.headers.update({
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:124.0) '
        'Gecko/20100101 Firefox/124.0'
    ),
    'Accept':          'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'fr-TN,fr;q=0.9,en-US;q=0.8,en;q=0.7',
    'Referer':         'https://www.ilboursa.com/',
    'Accept-Encoding': 'gzip, deflate',
    'Connection':      'keep-alive',
})
 
# Visit homepage first to receive session cookies
resp = SESSION.get('https://www.ilboursa.com/', timeout=15)
print(f"Homepage: HTTP {resp.status_code}")
time.sleep(3)  # short warm-up pause after homepage
 
 

Homepage: HTTP 200


In [15]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 4 — Core helpers                                   ║
# ╚══════════════════════════════════════════════════════════╝
 
def parse_date(date_str: str):
    """Convert '20/10/16 10:17' or '18/02/26 11:32' to a date object."""
    try:
        return datetime.strptime(date_str.strip().split()[0], '%d/%m/%y').date()
    except (ValueError, IndexError):
        try:
            return datetime.strptime(date_str.strip().split()[0], '%d/%m/%Y').date()
        except (ValueError, IndexError):
            return date.today()
 
 
def fetch_page(ilboursa_code: str, page: int) -> str | None:
    """
    Download one page of news for a stock.
 
    Page 1: ?s=CODE
    Page 2+: ?p=N&s=CODE
 
    Returns the raw HTML string, or None if the request failed.
    Retries once on 403.
    """
    if page == 1:
        url = f'https://www.ilboursa.com/marches/news_valeur?s={ilboursa_code}'
    else:
        url = f'https://www.ilboursa.com/marches/news_valeur?p={page}&s={ilboursa_code}'
 
    try:
        resp = SESSION.get(url, timeout=15)
 
        if resp.status_code == 403:
            time.sleep(30)
            resp = SESSION.get(url, timeout=15)
 
        if resp.status_code != 200:
            return None
 
        return resp.content.decode(resp.encoding or 'utf-8', errors='replace')
 
    except Exception:
        return None
 
 
def has_next_page(soup: BeautifulSoup, current_page: int) -> bool:
    """
    Two signals mean there is a next page:
    1. A link with text '>' exists (ilboursa's next arrow)
    2. A link with text str(current_page + 1) exists (visible page number)
    Either one is enough — return True if EITHER is found.
    """
    next_text = str(current_page + 1)
    
    for link in soup.find_all('a'):
        text = link.get_text(strip=True)
        if text in ('>', '»') or text == next_text:
            return True
    
    return False
def parse_articles_from_html(html: str, ticker: str) -> list:
    """
    Extract all articles from one page of HTML.
 
    Structure (flat — no wrapper divs):
        <span class="sp1">20/10/16 10:17</span>
        <a href="article-slug_12345">Article Title</a><br>
 
    Strategy:
        1. Find all <span class="sp1"> date elements
        2. Walk forward with .next_sibling to find the <a> after each span
        3. Extract title, href, date
    """
    soup = BeautifulSoup(html, 'html.parser')
    articles = []
 
    date_spans = soup.find_all('span', class_='sp1')
 
    for span in date_spans:
        date_text  = span.get_text(strip=True)
        published  = parse_date(date_text)
 
        # Walk forward through siblings to find the <a> tag
        a_tag  = None
        cursor = span.next_sibling
 
        for _ in range(10):
            if cursor is None:
                break
            if hasattr(cursor, 'name') and cursor.name == 'a':
                a_tag = cursor
                break
            cursor = cursor.next_sibling
 
        if a_tag is None:
            continue
 
        title = a_tag.get_text(strip=True)
        if not title or len(title) < 10:
            continue
 
        href = a_tag.get('href', '').strip()
        if not href:
            continue
 
        if href.startswith('http'):
            full_url = href
        else:
            safe_href = quote(href, safe='/-_.~')
            full_url  = urljoin('https://www.ilboursa.com/marches/', safe_href)
 
        articles.append({
            'ticker':       ticker,
            'isin_code':    None,
            'title':        title,
            'content':      '',
            'source':       'ilboursa.com',
            'url':          full_url,
            'published_at': str(published),
            'language':     'fr',
            'credibility':  0.90,
        })
 
    return articles, soup   # return soup too so caller can check has_next_page
 
 

In [16]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 5 — DB helpers                                     ║
# ╚══════════════════════════════════════════════════════════╝
 
def enrich_with_isin(articles: list, isin_map: dict) -> list:
    """Add isin_code to articles using a pre-loaded isin_map dict."""
    for a in articles:
        a['isin_code'] = isin_map.get(a['ticker'])
    return articles
 
 
def insert_articles(articles: list) -> int:
    """Bulk insert articles. Returns count of NEW rows inserted."""
    if not articles:
        return 0
 
    conn = get_conn()
    inserted = 0
    try:
        with conn.cursor() as cur:
            for a in articles:
                try:
                    cur.execute('''
                        INSERT INTO news_articles
                            (ticker, isin_code, title, content, source,
                             url, published_at, language, credibility)
                        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
                        ON CONFLICT (url) DO NOTHING
                    ''', (
                        a.get('ticker'),
                        a.get('isin_code'),
                        a.get('title', ''),
                        a.get('content', ''),
                        a.get('source', ''),
                        a.get('url', ''),
                        a.get('published_at'),
                        a.get('language', 'fr'),
                        a.get('credibility', 0.90),
                    ))
                    if cur.rowcount == 1:
                        inserted += 1
                except Exception:
                    continue
        conn.commit()
    finally:
        conn.close()
    return inserted
 
 
def load_isin_map() -> dict:
    """Load ticker -> isin_code mapping from company_metadata."""
    conn = get_conn()
    try:
        with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
            cur.execute('SELECT ticker, isin_code FROM company_metadata')
            return {row['ticker']: row['isin_code'] for row in cur.fetchall()}
    finally:
        conn.close()
 
isin_map = load_isin_map()
print(f"ISIN map loaded: {len(isin_map)} stocks")
 
 

ISIN map loaded: 70 stocks


In [17]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 6 — Scrape all pages for ONE stock                 ║
# ╚══════════════════════════════════════════════════════════╝
#
# This function scrapes every page for a single stock.
# It keeps fetching pages until has_next_page() returns False.
# Maximum pages cap (max_pages=50) is a safety net — no stock
# should ever have more than 50 pages on ilboursa.
 
def scrape_all_pages(ticker: str, ilboursa_code: str,
                     max_pages: int = 50,
                     delay: float = 1.5,
                     debug: bool = False) -> int:
    """
    Scrape ALL pages of news for one stock.
 
    Returns total number of NEW articles inserted into DB.
 
    How pagination works:
      - Fetch page 1, parse articles, check for ">" link
      - If ">" exists → fetch page 2, parse, check again
      - Repeat until ">" disappears (last page reached)
      - Safety cap: stop at max_pages regardless
 
    delay: seconds to wait between page requests (be polite)
    """
    total_inserted = 0
    page = 1
 
    while page <= max_pages:
        if debug:
            print(f"    Fetching page {page}...")
 
        # Download the page HTML
        html = fetch_page(ilboursa_code, page)
 
        if html is None:
            if debug:
                print(f"    Page {page} failed — stopping")
            break
 
        # Parse articles AND get the soup for pagination check
        articles, soup = parse_articles_from_html(html, ticker)
 
        if not articles and page == 1:
            # No articles on page 1 = wrong code or empty stock
            if debug:
                print(f"    No articles on page 1 — code may be wrong")
            break
 
        # Enrich and insert
        articles       = enrich_with_isin(articles, isin_map)
        new_count      = insert_articles(articles)
        total_inserted += new_count
 
        if debug:
            print(f"    Page {page}: {len(articles)} articles, {new_count} new")
 
        # Check if there is a next page
        if not has_next_page(soup, page):
            if debug:
                print(f"    No '>' found — last page reached at page {page}")
            break
 
        page += 1
        time.sleep(delay)  # polite pause between page requests
 
    return total_inserted
 
 
# ── QUICK TEST: scrape all pages for AMEN BANK ────────────────────────────
print("Testing full pagination with AMEN BANK (11 pages expected)...")
n = scrape_all_pages('AMEN BANK', 'AB', debug=True)
print(f"\nAMEN BANK: {n} new articles inserted")
 
# Verify in DB
conn = get_conn()
count = pd.read_sql(
    "SELECT COUNT(*) as n FROM news_articles WHERE ticker='AMEN BANK'",
    conn
).iloc[0]['n']
conn.close()
print(f"AMEN BANK articles now in DB: {count}")
# Expected: ~220 (11 pages × 20 articles)
 
 

Testing full pagination with AMEN BANK (11 pages expected)...
    Fetching page 1...
    Page 1: 20 articles, 20 new
    Fetching page 2...
    Page 2: 20 articles, 20 new
    Fetching page 3...
    Page 3: 20 articles, 20 new
    Fetching page 4...
    Page 4: 20 articles, 20 new
    Fetching page 5...
    Page 5: 20 articles, 20 new
    Fetching page 6...
    Page 6: 20 articles, 20 new
    Fetching page 7...
    Page 7: 20 articles, 20 new
    Fetching page 8...
    Page 8: 20 articles, 20 new
    Fetching page 9...
    Page 9: 20 articles, 20 new
    Fetching page 10...
    Page 10: 20 articles, 20 new
    Fetching page 11...
    Page 11: 20 articles, 20 new
    No '>' found — last page reached at page 11

AMEN BANK: 220 new articles inserted
AMEN BANK articles now in DB: 220


C:\Users\Negza\AppData\Local\Temp\ipykernel_4964\2109590420.py:78: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  count = pd.read_sql(


In [ ]:
# Diagnostic — check pagination links on page 9
html = fetch_page('AB', 9)
soup = BeautifulSoup(html, 'html.parser')

print("Short links on page 9:")
for link in soup.find_all('a'):
    text = link.get_text(strip=True)
    if len(text) <= 3 and text:
        print(f"  text={repr(text)}  href={repr(link.get('href','')[:50])}")

In [19]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 7 — Full historical scrape for ALL 68 stocks       ║
# ╚══════════════════════════════════════════════════════════╝
#
# TIMING ESTIMATE:
# Each stock has on average ~5 pages (100 articles).
# Each page takes ~1.5s + network time ≈ ~2s per page.
# 68 stocks × 5 pages × 2s = ~11 minutes total.
# Stocks with many pages (banks) take longer.
# The 30s pause every 15 stocks adds ~6 minutes.
# Total estimate: 15-20 minutes.
#
# DO NOT close Jupyter or put your computer to sleep during this.
 
def run_full_historical_scrape():
    """Scrape all pages for all 68 stocks."""
 
    print("=" * 65)
    print("BVMT Full Historical News Scrape — ilboursa.com")
    print("=" * 65)
    print(f"Stocks: {len(ILBOURSA_TICKERS)}")
    print(f"Delay between pages: 1.5s | Delay between stocks: 3s")
    print(f"Estimated time: 15-20 minutes")
    print("=" * 65)
    print()
 
    grand_total   = 0
    zero_stocks   = []
    stock_summary = []
 
    for i, (ticker, code) in enumerate(ILBOURSA_TICKERS.items(), 1):
        print(f"[{i:2d}/{len(ILBOURSA_TICKERS)}] {ticker:<30} (s={code})", end=" ", flush=True)
 
        inserted = scrape_all_pages(
            ticker,
            code,
            max_pages=50,
            delay=1.5,
            debug=False      # set True to see page-by-page detail
        )
 
        grand_total += inserted
        stock_summary.append({'ticker': ticker, 'inserted': inserted})
 
        if inserted == 0:
            zero_stocks.append(ticker)
            print(f"→ 0 articles")
        else:
            print(f"→ {inserted} new articles")
 
        # 3 second pause between stocks
        time.sleep(3)
 
        # Longer pause every 15 stocks to avoid rate limiting
        if i % 15 == 0:
            print(f"  [60s cooldown pause after {i} stocks...]")
            time.sleep(60)
 
    # ── Final summary ─────────────────────────────────────────
    print()
    print("=" * 65)
    print("FULL HISTORICAL SCRAPE COMPLETE")
    print("=" * 65)
    print(f"  Total new articles inserted: {grand_total:,}")
    print(f"  Stocks with articles:        {len(ILBOURSA_TICKERS) - len(zero_stocks)}")
    print(f"  Stocks with 0 articles:      {len(zero_stocks)}")
 
    if zero_stocks:
        print()
        print("  Stocks with 0 articles:")
        for s in zero_stocks:
            print(f"    {s}")
 
    # Top 10 most covered stocks
    print()
    print("  Top 10 most covered stocks:")
    top10 = sorted(stock_summary, key=lambda x: x['inserted'], reverse=True)[:10]
    for s in top10:
        print(f"    {s['ticker']:<30} {s['inserted']:>5} articles")
 
    return grand_total
 
 
# Run the full historical scrape
# Make sure you ran TRUNCATE TABLE news_articles RESTART IDENTITY
# in pgAdmin before running this cell.
total = run_full_historical_scrape()
 
 

BVMT Full Historical News Scrape — ilboursa.com
Stocks: 68
Delay between pages: 1.5s | Delay between stocks: 3s
Estimated time: 15-20 minutes

[ 1/68] AMEN BANK                      (s=AB) → 0 articles
[ 2/68] BIAT                           (s=BIAT) → 280 new articles
[ 3/68] ATTIJARI BANK                  (s=TJARI) → 220 new articles
[ 4/68] BH                             (s=BH) → 200 new articles
[ 5/68] BNA                            (s=BNA) → 200 new articles
[ 6/68] BT                             (s=BT) → 140 new articles
[ 7/68] STB                            (s=STB) → 200 new articles
[ 8/68] UIB                            (s=UIB) → 200 new articles
[ 9/68] ATB                            (s=ATB) → 140 new articles
[10/68] UBCI                           (s=UBCI) → 200 new articles
[11/68] WIFACK INT BANK                (s=WIFAK) → 160 new articles
[12/68] STAR                           (s=STAR) → 100 new articles
[13/68] ASTREE                         (s=AST) → 55 new articles
[1

In [20]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 8 — Verify final DB state                          ║
# ╚══════════════════════════════════════════════════════════╝
 
conn = get_conn()
 
# Articles per stock with date range
df = pd.read_sql('''
    SELECT
        ticker,
        COUNT(*)                    AS articles,
        MIN(published_at)::text     AS oldest,
        MAX(published_at)::text     AS newest
    FROM news_articles
    GROUP BY ticker
    ORDER BY articles DESC
''', conn)
 
print(f"Total stocks with articles: {len(df)}")
print(f"Total articles in DB: {df['articles'].sum():,}")
print()
print(df.to_string(index=False))
 
# Year distribution — how many articles per year
df_years = pd.read_sql('''
    SELECT
        EXTRACT(YEAR FROM published_at)::int AS year,
        COUNT(*) AS articles
    FROM news_articles
    GROUP BY year
    ORDER BY year
''', conn)
 
print()
print("Articles by year:")
print(df_years.to_string(index=False))
 
conn.close()

Total stocks with articles: 68
Total articles in DB: 7,937

            ticker  articles     oldest     newest
              BIAT       280 2016-10-07 2026-03-04
          TUNISAIR       280 2014-02-21 2026-03-11
         AMEN BANK       220 2016-10-20 2026-02-18
     ATTIJARI BANK       220 2015-12-08 2026-02-27
POULINA GP HOLDING       200 2015-06-05 2026-02-20
               BNA       200 2015-10-21 2026-01-31
               UIB       200 2017-01-18 2026-02-06
                BH       200 2016-07-30 2026-01-26
         CITY CARS       200 2014-03-07 2026-01-21
               STB       200 2015-04-14 2026-01-22
              UBCI       200 2015-05-18 2026-01-20
              SFBT       180 2014-12-19 2026-01-31
   WIFACK INT BANK       160 2015-06-05 2026-03-06
  ONE TECH HOLDING       160 2014-05-16 2026-02-25
    TELNET HOLDING       160 2014-05-21 2026-01-16
               SAH       160 2015-04-21 2026-02-11
              UADH       160 2015-11-27 2024-11-28
   MAGASIN GENERAL    

C:\Users\Negza\AppData\Local\Temp\ipykernel_4964\3537705204.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql('''
C:\Users\Negza\AppData\Local\Temp\ipykernel_4964\3537705204.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_years = pd.read_sql('''
